GROUP NAME: Cognitive Crew

TEAM MEMBERS:
  1. Lanka Devi Satwika - B24DS013
  2. Kotapati Sai Mounika - B24CS019
  3. Bailapudi Kusuma Teja - B24CS011
  4. Bodike Chaithali - B24CS013
  5. Jayasurya Boorada

# Problem 2: Campus Graph — Informed Search

## Greedy Best-First Search and A* Search

### 1.Objective

Find a path from the campus location **A** to the destination **F** using:

1. Greedy Best-First Search
2. A* Search

The two algorithms will be compared using path cost, nodes expanded, execution time, optimality, and heuristic effectiveness.

This implementation follows the graph, edge costs, heuristic values, and required outputs specified in the project statement.

## 2. Input Graph

The given campus graph contains 6 nodes and 7 bidirectional edges.

| Edge | Cost |
|---|---:|
| A – B | 4 |
| A – C | 2 |
| B – D | 5 |
| C – D | 3 |
| C – E | 6 |
| D – E | 3 |
| E – F | 2 |

In [1]:
import heapq
import time
import pandas as pd

In [2]:
graph = {
    'A': [('B', 4), ('C', 2)],
    'B': [('A', 4), ('D', 5)],
    'C': [('A', 2), ('D', 3), ('E', 6)],
    'D': [('B', 5), ('C', 3), ('E', 3)],
    'E': [('C', 6), ('D', 3), ('F', 2)],
    'F': [('E', 2)]
}

start = 'A'
goal = 'F'

print("Start:", start)
print("Goal:", goal)
print("\nGraph:")
for node, neighbours in graph.items():
    print(node, "->", neighbours)

Start: A
Goal: F

Graph:
A -> [('B', 4), ('C', 2)]
B -> [('A', 4), ('D', 5)]
C -> [('A', 2), ('D', 3), ('E', 6)]
D -> [('B', 5), ('C', 3), ('E', 3)]
E -> [('C', 6), ('D', 3), ('F', 2)]
F -> [('E', 2)]


## 3. Heuristic Values

The heuristic values given in the project statement are:

| Node | h(n) |
|---|---:|
| A | 7 |
| B | 8 |
| C | 5 |
| D | 4 |
| E | 2 |
| F | 0 |

The goal node has heuristic value 0.

In [3]:
heuristic = {
    'A': 7,
    'B': 8,
    'C': 5,
    'D': 4,
    'E': 2,
    'F': 0
}

print("Heuristic Values:")
for node, h in heuristic.items():
    print(f"h({node}) = {h}")

Heuristic Values:
h(A) = 7
h(B) = 8
h(C) = 5
h(D) = 4
h(E) = 2
h(F) = 0


## 4. Greedy Best-First Search

Greedy Best-First Search selects the node that appears closest to the goal according to the heuristic.

### Evaluation function

**f(n) = h(n)**

It considers only the estimated remaining cost and does not consider the path cost already travelled.

### Main idea

1. Start from A.
2. Put the start node in the priority queue.
3. Select the node with the smallest heuristic value.
4. Expand that node and add its unvisited neighbours.
5. Continue until F is reached.
6. Reconstruct the path and calculate its total cost.

In [4]:
import heapq
import time

def reconstruct_path(parent, goal):
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


def calculate_path_cost(graph, path):
    total = 0

    for u, v in zip(path, path[1:]):
        for neighbour, cost in graph[u]:
            if neighbour == v:
                total += cost
                break

    return total

### Greedy Best-First Search Function

In [5]:
def greedy_best_first_search(graph, heuristic, start, goal):
    counter = 0
    priority_queue = [(heuristic[start], counter, start)]

    visited = set()
    parent = {start: None}

    nodes_expanded = 0

    while priority_queue:
        _, _, current = heapq.heappop(priority_queue)

        if current in visited:
            continue

        visited.add(current)

        if current == goal:
            path = reconstruct_path(parent, goal)
            cost = calculate_path_cost(graph, path)
            return {
                "path_found": True,
                "path": path,
                "cost": cost,
                "nodes_expanded": nodes_expanded
            }

        nodes_expanded += 1

        for neighbour, _ in graph[current]:
            if neighbour not in visited:
                if neighbour not in parent:
                    parent[neighbour] = current

                counter += 1
                heapq.heappush(
                    priority_queue,
                    (heuristic[neighbour], counter, neighbour)
                )

    return {
        "path_found": False,
        "path": [],
        "cost": None,
        "nodes_expanded": nodes_expanded
    }

## 5. Test Greedy Best-First Search

In [6]:
start_time = time.perf_counter()

greedy_result = greedy_best_first_search(
    graph, heuristic, start, goal
)

greedy_time = time.perf_counter() - start_time

print("Algorithm: Greedy Best-First Search")
print("Path Found:", "Yes" if greedy_result["path_found"] else "No")
print("Path:", " → ".join(greedy_result["path"]) if greedy_result["path"] else "None")
print("Total Path Cost:", greedy_result["cost"])
print("Nodes Expanded:", greedy_result["nodes_expanded"])
print(f"Execution Time: {greedy_time:.8f} seconds")

Algorithm: Greedy Best-First Search
Path Found: Yes
Path: A → C → E → F
Total Path Cost: 10
Nodes Expanded: 3
Execution Time: 0.00011705 seconds


## 6. Observation — Greedy Best-First Search

Greedy Search chooses the next node using **only h(n)**.

Therefore, it can prefer a node that looks closer to the goal even when reaching that node requires a larger total path cost.

This is the main limitation that will be examined using the second graph.

## 7. A* Search

A* combines:

- **g(n):** actual cost from the start node to the current node
- **h(n):** estimated cost from the current node to the goal

### Evaluation function

**f(n) = g(n) + h(n)**

Unlike Greedy Search, A* considers both the cost already travelled and the estimated remaining cost.

### Main idea

1. Start from A with g(A) = 0.
2. Calculate f(n) = g(n) + h(n).
3. Select the node with the smallest f(n).
4. Update a node when a cheaper path to it is found.
5. Continue until F is reached.
6. Reconstruct the final path and calculate its total cost.

In [7]:
def a_star_search(graph, heuristic, start, goal):
    counter = 0

    g_cost = {start: 0}
    parent = {start: None}

    priority_queue = [(heuristic[start], counter, start)]

    closed = set()
    nodes_expanded = 0

    while priority_queue:
        f_value, _, current = heapq.heappop(priority_queue)

        if current in closed:
            continue

        if current == goal:
            path = reconstruct_path(parent, goal)
            cost = calculate_path_cost(graph, path)

            return {
                "path_found": True,
                "path": path,
                "cost": cost,
                "nodes_expanded": nodes_expanded
            }

        closed.add(current)
        nodes_expanded += 1

        for neighbour, edge_cost in graph[current]:
            new_g = g_cost[current] + edge_cost

            if neighbour not in g_cost or new_g < g_cost[neighbour]:
                g_cost[neighbour] = new_g
                parent[neighbour] = current

                counter += 1
                f = new_g + heuristic[neighbour]

                heapq.heappush(
                    priority_queue,
                    (f, counter, neighbour)
                )

    return {
        "path_found": False,
        "path": [],
        "cost": None,
        "nodes_expanded": nodes_expanded
    }

## 8. Test A* Search

In [8]:
start_time = time.perf_counter()

astar_result = a_star_search(
    graph, heuristic, start, goal
)

astar_time = time.perf_counter() - start_time

print("Algorithm: A* Search")
print("Path Found:", "Yes" if astar_result["path_found"] else "No")
print("Path:", " → ".join(astar_result["path"]) if astar_result["path"] else "None")
print("Total Path Cost:", astar_result["cost"])
print("Nodes Expanded:", astar_result["nodes_expanded"])
print(f"Execution Time: {astar_time:.8f} seconds")

Algorithm: A* Search
Path Found: Yes
Path: A → C → E → F
Total Path Cost: 10
Nodes Expanded: 4
Execution Time: 0.00022414 seconds


## 9. Comparison — Given Campus Graph

In [9]:
results_main = pd.DataFrame([
    {
        "Algorithm": "Greedy Best-First Search",
        "Path Found": "Yes" if greedy_result["path_found"] else "No",
        "Path": " → ".join(greedy_result["path"]),
        "Path Cost": greedy_result["cost"],
        "Nodes Expanded": greedy_result["nodes_expanded"],
        "Execution Time (s)": greedy_time
    },
    {
        "Algorithm": "A* Search",
        "Path Found": "Yes" if astar_result["path_found"] else "No",
        "Path": " → ".join(astar_result["path"]),
        "Path Cost": astar_result["cost"],
        "Nodes Expanded": astar_result["nodes_expanded"],
        "Execution Time (s)": astar_time
    }
])

results_main

,Algorithm,Path Found,Path,Path Cost,Nodes Expanded,Execution Time (s)
0,Greedy Best-First Search,Yes,A → C → E → F,10,3,0.000117
1,A* Search,Yes,A → C → E → F,10,4,0.000224


### Observation

The table above gives the actual performance obtained from the implementation.

**Important:** execution time is machine-dependent and can change slightly between runs.

The project statement gives the sample route **A → C → D → E → F** with total cost 10. However, with the stated Greedy rule **f(n)=h(n)** and the supplied heuristic values, Greedy may select E before D because h(E)=2 is smaller than h(D)=4. Therefore, the implementation follows the stated search function rather than hard-coding the sample route.

The path cost and nodes expanded should be discussed using the actual output produced by the notebook.

## 10. Second Graph Experiment

The project specifically requires a second graph where Greedy Best-First Search and A* are likely to choose different routes.

The purpose is to demonstrate that **Greedy Search does not necessarily produce the optimal path**.

We use the following graph:

- A → B = 1
- B → F = 100
- A → C = 5
- C → D = 5
- D → F = 5

Two possible routes are:

**A → B → F = 101**

**A → C → D → F = 15**

The heuristic values are chosen so that B appears very attractive to Greedy Search.

In [10]:
graph2 = {
    'A': [('B', 1), ('C', 5)],
    'B': [('A', 1), ('F', 100)],
    'C': [('A', 5), ('D', 5)],
    'D': [('C', 5), ('F', 5)],
    'F': [('B', 100), ('D', 5)]
}

heuristic2 = {
    'A': 15,
    'B': 1,
    'C': 10,
    'D': 5,
    'F': 0
}

start2 = 'A'
goal2 = 'F'

print("Second Graph:")
for node, neighbours in graph2.items():
    print(node, "->", neighbours)

print("\nHeuristic Values:")
for node, h in heuristic2.items():
    print(f"h({node}) = {h}")

Second Graph:
A -> [('B', 1), ('C', 5)]
B -> [('A', 1), ('F', 100)]
C -> [('A', 5), ('D', 5)]
D -> [('C', 5), ('F', 5)]
F -> [('B', 100), ('D', 5)]

Heuristic Values:
h(A) = 15
h(B) = 1
h(C) = 10
h(D) = 5
h(F) = 0


### Expected Route Analysis

For Greedy Search:

- From A, B has h(B)=1.
- C has h(C)=10.
- Therefore Greedy prefers B.
- It then reaches F through the expensive edge B–F.

For A*:

- The cost already travelled, g(n), is also considered.
- The cheaper route through C and D has total cost 15.
- Therefore A* should find the lower-cost route.

In [11]:
start_time = time.perf_counter()

greedy_result2 = greedy_best_first_search(
    graph2, heuristic2, start2, goal2
)

greedy_time2 = time.perf_counter() - start_time

print("Algorithm: Greedy Best-First Search")
print("Path Found:", "Yes" if greedy_result2["path_found"] else "No")
print("Path:", " → ".join(greedy_result2["path"]))
print("Total Path Cost:", greedy_result2["cost"])
print("Nodes Expanded:", greedy_result2["nodes_expanded"])
print(f"Execution Time: {greedy_time2:.8f} seconds")

Algorithm: Greedy Best-First Search
Path Found: Yes
Path: A → B → F
Total Path Cost: 101
Nodes Expanded: 2
Execution Time: 0.00014942 seconds


In [12]:
start_time = time.perf_counter()

astar_result2 = a_star_search(
    graph2, heuristic2, start2, goal2
)

astar_time2 = time.perf_counter() - start_time

print("Algorithm: A* Search")
print("Path Found:", "Yes" if astar_result2["path_found"] else "No")
print("Path:", " → ".join(astar_result2["path"]))
print("Total Path Cost:", astar_result2["cost"])
print("Nodes Expanded:", astar_result2["nodes_expanded"])
print(f"Execution Time: {astar_time2:.8f} seconds")

Algorithm: A* Search
Path Found: Yes
Path: A → C → D → F
Total Path Cost: 15
Nodes Expanded: 4
Execution Time: 0.00022314 seconds


## 11. Second Graph Comparison

In [13]:
results_second = pd.DataFrame([
    {
        "Algorithm": "Greedy Best-First Search",
        "Path": " → ".join(greedy_result2["path"]),
        "Path Cost": greedy_result2["cost"],
        "Nodes Expanded": greedy_result2["nodes_expanded"],
        "Execution Time (s)": greedy_time2
    },
    {
        "Algorithm": "A* Search",
        "Path": " → ".join(astar_result2["path"]),
        "Path Cost": astar_result2["cost"],
        "Nodes Expanded": astar_result2["nodes_expanded"],
        "Execution Time (s)": astar_time2
    }
])

results_second

,Algorithm,Path,Path Cost,Nodes Expanded,Execution Time (s)
0,Greedy Best-First Search,A → B → F,101,2,0.000149
1,A* Search,A → C → D → F,15,4,0.000223


### Observation — Second Graph

The second graph demonstrates the key difference between the two algorithms.

Greedy Best-First Search focuses on **h(n)** and therefore follows the route that looks closer to the goal.

A* considers **g(n)+h(n)**, so it accounts for the cost already travelled.

The optimal route in this graph is:

**A → C → D → F**

with total cost:

**5 + 5 + 5 = 15**

The Greedy route is:

**A → B → F**

with total cost:

**1 + 100 = 101**

Therefore, this experiment demonstrates that Greedy Best-First Search is **not guaranteed to return an optimal path**, while A* can find the optimal path when its heuristic satisfies the required conditions.

## 12. Overall Comparison

### Greedy Best-First Search

- Uses **f(n)=h(n)**.
- Considers only the estimated distance to the goal.
- Can be fast because it focuses strongly on the goal.
- It is **not guaranteed to find the optimal path**.

### A* Search

- Uses **f(n)=g(n)+h(n)**.
- Considers both the cost already travelled and the estimated remaining cost.
- Usually explores nodes more intelligently than uninformed search.
- With an appropriate admissible heuristic, A* can guarantee an optimal solution.

In [14]:
overall_comparison = pd.DataFrame({
    "Criterion": [
        "Evaluation Function",
        "Uses Actual Cost g(n)",
        "Uses Heuristic h(n)",
        "Optimality",
        "Main Advantage",
        "Main Limitation"
    ],
    "Greedy Best-First Search": [
        "f(n) = h(n)",
        "No",
        "Yes",
        "Not guaranteed",
        "Goal-directed search",
        "Can choose an expensive route"
    ],
    "A* Search": [
        "f(n) = g(n) + h(n)",
        "Yes",
        "Yes",
        "Optimal with an appropriate admissible heuristic",
        "Balances actual and estimated cost",
        "May explore more states / require more memory"
    ]
})

overall_comparison

,Criterion,Greedy Best-First Search,A* Search
0,Evaluation Function,f(n) = h(n),f(n) = g(n) + h(n)
1,Uses Actual Cost g(n),No,Yes
2,Uses Heuristic h(n),Yes,Yes
3,Optimality,Not guaranteed,Optimal with an appropriate admissible heuristic
4,Main Advantage,Goal-directed search,Balances actual and estimated cost
5,Main Limitation,Can choose an expensive route,May explore more states / require more memory


## 13. Final Observations

1. Both Greedy Best-First Search and A* use heuristic information to guide the search.
2. Greedy uses only **h(n)**, whereas A* uses **g(n)+h(n)**.
3. Greedy can reach the goal quickly but may select a higher-cost path.
4. A* balances the cost already travelled with the estimated cost to the goal.
5. The second graph clearly demonstrates the non-optimality of Greedy Search.
6. The effectiveness of a heuristic strongly affects informed search performance.
7. Execution time should be compared using the actual values produced by the program because it depends on the machine and runtime environment.

## 14. Conclusion

Greedy Best-First Search and A* are informed search algorithms that use heuristic information to guide path finding.

Greedy Best-First Search selects nodes using only the estimated cost **h(n)**, making it strongly goal-directed but not necessarily optimal.

A* uses **g(n)+h(n)**, combining the actual path cost and estimated remaining cost. The second graph demonstrates why considering both values is important: Greedy can select a route with cost 101, while A* can identify the optimal route with cost 15.

Thus, A* provides a better balance between path cost and heuristic guidance when an appropriate heuristic is available.